In [ ]:
# =============================================================================
# GNN Pipeline for Anti-inflammatory Activity Prediction (FINAL CLEAN VERSION)
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, GATv2Conv, TransformerConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_undirected
from torch_geometric.nn import GraphNorm

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import numpy as np
import pandas as pd
from rdkit import RDLogger, Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import *
from scipy import stats
import warnings
import json
from datetime import datetime
import copy
import optuna

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(r"D:\Biotechnology Research\WholedatasetGAN.csv")
df["active"] = (df["pIC50"] >= 6).astype(int)
df = df.dropna(subset=["pIC50"]).reset_index(drop=True)
df = df.drop_duplicates(subset="SMILES").reset_index(drop=True)
print("Dataset size:", len(df))
print("\nClass distribution:")
print(df['active'].value_counts(normalize=True))
torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

NameError: name 'pd' is not defined

In [ ]:
# =============================================================================
# ATOM FEATURES
# =============================================================================
def enhanced_atom_features(atom):
    # Atom type one-hot (common elements)
    atom_types = ['C','N','O','S','F','Cl','Br','I','P','B','Unknown']
    symbol = atom.GetSymbol()
    features = [1.0 if symbol == t else 0.0 for t in atom_types[:-1]]

    if symbol not in atom_types[:-1]:
        features.append(1.0)
    else:
        features.append(0.0)
    
    # Scalar features
    features += [
        atom.GetAtomicNum() / 100.0,
        atom.GetDegree() / 8.0,
        atom.GetFormalCharge() / 5.0,
        atom.GetTotalNumHs() / 4.0,
        atom.GetTotalValence() / 8.0,
        float(atom.GetIsAromatic()),
        float(atom.IsInRing()),
        float(atom.GetChiralTag() != Chem.rdchem.ChiralType.CHI_UNSPECIFIED)  # chirality
    ]
    
    # Hybridization one-hot
    hyb = atom.GetHybridization()
    hyb_types = [Chem.rdchem.HybridizationType.SP,
                 Chem.rdchem.HybridizationType.SP2,
                 Chem.rdchem.HybridizationType.SP3]
    features += [1.0 if hyb == h else 0.0 for h in hyb_types]
    
    return features

def enhanced_bond_features(bond):
    bt = bond.GetBondType()
    # Bond type one-hot
    features = [
        float(bt == Chem.rdchem.BondType.SINGLE),
        float(bt == Chem.rdchem.BondType.DOUBLE),
        float(bt == Chem.rdchem.BondType.TRIPLE),
        float(bt == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated()),
        float(bond.IsInRing())
    ]
    features.append(float(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE))
    return features


In [ ]:
# =============================================================================
# SMILES → GRAPH
# =============================================================================
def smiles_to_graph(smiles, label, pic50):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Node features
    x = [enhanced_atom_features(a) for a in mol.GetAtoms()]
    
    # Edge features
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        f = enhanced_bond_features(bond)
        edge_index.append([i, j])
        edge_attr.append(f)
    
    if len(edge_index) == 0:
        edge_index = [[0, 0]]
        edge_attr = [[0]*7]  # adjust dimension
    
    # Convert to tensor and ensure undirected
    edge_index = torch.tensor(edge_index).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    edge_index, edge_attr = to_undirected(edge_index, edge_attr)
    # Descriptors and fingerprints (optional, can be omitted if not needed)
    desc = [
        Descriptors.MolWt(mol)/1000,
        Descriptors.MolLogP(mol)/10,
        Descriptors.TPSA(mol)/100,
        Descriptors.NumRotatableBonds(mol)/20,
        QED.qed(mol)
    ]
    fp = np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
    
    data = Data(
        x=torch.tensor(x, dtype=torch.float),
        edge_index=edge_index,
        edge_attr=edge_attr,
        fp=torch.tensor(fp, dtype=torch.float).unsqueeze(0),
        desc=torch.tensor(desc, dtype=torch.float).unsqueeze(0),
        y_cls=torch.tensor([label], dtype=torch.float),
        y_reg=torch.tensor([pic50], dtype=torch.float),
        reg_mask=torch.tensor([1.0], dtype=torch.float),
        smiles=smiles  # store for later augmentation
    )
    return data

Graphs: 4300


In [ ]:
# =============================================================================
# BUILD GRAPH DATASET
# =============================================================================
graphs = []
valid_smiles = []
for _, row in df.iterrows():
    g = smiles_to_graph(row.SMILES, row.active, row.pIC50)
    if g is not None:
        graphs.append(g)
        valid_smiles.append(row.SMILES)

df_filtered = df[df["SMILES"].isin(valid_smiles)].reset_index(drop=True)
print("Graphs:", len(graphs))



--- Chemical space analysis on full dataset ---
Tanimoto analysis (sample 500 molecules):
  Mean similarity: 0.1225 ± 0.0514
  Diversity (1-mean): 0.8775


(np.float64(0.122498824613658),
 np.float64(0.051417260619602854),
 np.float64(0.877501175386342))

In [ ]:
# =============================================================================
# 4. CHEMICAL SPACE ANALYSIS (TANIMOTO)
# =============================================================================
def tanimoto_analysis(smiles_list, sample_size=500):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s)]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in mols]
    if len(fps) > sample_size:
        idx = np.random.choice(len(fps), sample_size, replace=False)
        fps = [fps[i] for i in idx]
    sims = []
    for i in range(len(fps)):
        for j in range(i+1, len(fps)):
            sims.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
    mean_sim = np.mean(sims)
    std_sim = np.std(sims)
    diversity = 1 - mean_sim
    print(f"Tanimoto analysis (sample {len(fps)} molecules):")
    print(f"  Mean similarity: {mean_sim:.4f} ± {std_sim:.4f}")
    print(f"  Diversity (1-mean): {diversity:.4f}")
    return mean_sim, std_sim, diversity

print("\n--- Chemical space analysis on full dataset ---")
tanimoto_analysis(df_filtered['SMILES'].tolist())

In [ ]:
# =============================================================================
# MSMP LAYER
# =============================================================================

class UnifiedMSMP(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=256, heads=8, dropout=0.2):
        super().__init__()
        self.node_embed = nn.Linear(node_dim, hidden)
        self.edge_proj = nn.Linear(edge_dim, 32)  # for attention layers
        self.edge_proj_gine = nn.Linear(edge_dim, hidden)   # for GINEConv
        self.norm_local = GraphNorm(hidden)
        self.norm_attn = GraphNorm(hidden)
        self.norm_trans = GraphNorm(hidden)

        # Local (edge-aware GCN)
        self.conv_local = GINEConv(nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)
        ))
        # Attention
        self.conv_attn = GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
        # Transformer (long-range)
        self.conv_trans = TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32)

        self.norm = nn.LayerNorm(hidden * 6)
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 3 * 2, 256),   # mean+max pooled
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128)
        )
        self.cls_head = nn.Linear(128, 1)
        self.reg_head = nn.Linear(128, 1)

    def forward(self, data):
        x = self.node_embed(data.x)
        e_attn = self.edge_proj(data.edge_attr)
        e_gine = self.edge_proj_gine(data.edge_attr)

        # Local (edge‑aware) path
        h_local = self.conv_local(x, data.edge_index, e_gine)
        h_local = self.norm_local(h_local, data.batch)
        h_local = F.relu(h_local)

        # Attention path
        h_attn = self.conv_attn(x, data.edge_index, e_attn)
        h_attn = self.norm_attn(h_attn, data.batch)
        h_attn = F.relu(h_attn)

        # Transformer path
        h_trans = self.conv_trans(x, data.edge_index, e_attn)
        h_trans = self.norm_trans(h_trans, data.batch)
        h_trans = F.relu(h_trans)

        # Concatenate multi‑scale representations
        h = torch.cat([h_local, h_attn, h_trans], dim=-1)

        # Global pooling
        h_mean = global_mean_pool(h, data.batch)
        h_max = global_max_pool(h, data.batch)
        h_pool = torch.cat([h_mean, h_max], dim=-1)
        h_pool = self.norm(h_pool)          # LayerNorm

        # Fusion MLP
        feat = self.fusion(h_pool)

        # Multi‑task outputs
        return self.cls_head(feat).squeeze(-1), self.reg_head(feat).squeeze(-1)

In [ ]:
# =============================================================================
# FOCAL LOSS (improves classification accuracy)
# =============================================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

In [ ]:
# %%
# =============================================================================
# TRAINING WITH DUAL AUGMENTATION (EDGE + FEATURE DROPOUT)
# =============================================================================

def smiles_randomization(smiles, n_random=1):
    """Generate n_random randomized SMILES strings for a given SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    randomized = []
    for _ in range(n_random):
        try:
            rand_smiles = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            randomized.append(rand_smiles)
        except:
            continue
    return randomized

def edge_dropout(data, p):
    data = data.clone()
    edge_index = data.edge_index
    edge_attr = data.edge_attr
    num_edges = edge_index.size(1) // 2
    mask = torch.rand(num_edges) > p
    mask = mask.repeat_interleave(2)
    data.edge_index = edge_index[:, mask]
    data.edge_attr = edge_attr[mask]
    return data

def feature_masking(data, p):
    data = data.clone()
    x = data.x.clone()
    mask = torch.rand(x.size(0), 1, device=x.device) > p
    data.x = x * mask.float()
    return data

In [ ]:
# =============================================================================
# TRAINING WITH EARLY STOPPING
# =============================================================================
def train_model_with_early_stopping(model, train_loader, val_loader, device,
                                    epochs=200, patience=25, augment=False,
                                    alpha=0.75, gamma=2, reg_weight=0.1, lr=5e-4,
                                    edge_dropout_p=0.1, feat_mask_p=0.1):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20)
    
    cls_loss_fn = FocalLoss(alpha=alpha, gamma=gamma)
    reg_loss_fn = nn.MSELoss()

    best_auc = 0
    best_thr = 0.5
    wait = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            if augment:
                batch = edge_dropout(batch, p=edge_dropout_p)
                batch = feature_masking(batch, p=feat_mask_p)
            optimizer.zero_grad()
            c, r = model(batch)
            loss = cls_loss_fn(c, batch.y_cls.squeeze()) + reg_weight * reg_loss_fn(r, batch.y_reg_norm.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        probs, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                c, _ = model(batch)
                probs.extend(torch.sigmoid(c).cpu().numpy())
                labels.extend(batch.y_cls.cpu().numpy())
        auc = roc_auc_score(labels, probs)
        if auc > best_auc:
            best_auc = auc
            fpr, tpr, thr = roc_curve(labels, probs)
            idx = np.argmax(tpr - fpr)
            best_thr = thr[idx]
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_thr


In [ ]:
# =============================================================================

# 7. EVALUATION FUNCTION (returns all metrics)

# =============================================================================

def evaluate_model(model, test_loader, device, threshold):
    model.eval()
    all_probs, all_labels = [], []
    all_reg_preds, all_reg_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            cls_out, reg_out = model(batch)

            probs = torch.sigmoid(cls_out).cpu().numpy()
            labels = batch.y_cls.cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(labels)

            mask = batch.reg_mask.bool().cpu()

            if mask.sum() > 0:
                all_reg_preds.extend(reg_out[mask].cpu().numpy())
                all_reg_labels.extend(batch.y_reg[mask].cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    preds = (all_probs >= threshold).astype(int)

# ===============================
# Classification Metrics
# ===============================

    cm = confusion_matrix(all_labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    acc = accuracy_score(all_labels, preds)

# Safe AUC calculation
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5

    mcc = matthews_corrcoef(all_labels, preds)
    bal_acc = balanced_accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds)
    precision = precision_score(all_labels, preds)
    recall = recall_score(all_labels, preds)

    cls_results = {
        'accuracy': acc,
        'auc': auc,
        'mcc': mcc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'balanced_accuracy': bal_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'threshold': threshold
    }

# ===============================
# Regression Metrics
# ===============================

    if len(all_reg_labels) > 0:
        all_reg_preds = np.array(all_reg_preds)
        all_reg_labels = np.array(all_reg_labels)

        r2 = r2_score(all_reg_labels, all_reg_preds)
        rmse = np.sqrt(mean_squared_error(all_reg_labels, all_reg_preds))
        mae = mean_absolute_error(all_reg_labels, all_reg_preds)

        reg_results = {
            'r2': r2,
            'rmse': rmse,
            'mae': mae
        }
    else:
        reg_results = {}

    return cls_results, reg_results


In [ ]:
# =============================================================================
# SIMILARITY ANALYSIS PER FOLD
# =============================================================================

def fold_similarity(train_smiles, test_smiles, nbits=2048, radius=2):
    """Compute mean max Tanimoto similarity between test and train."""
    train_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius, nbits) for s in train_smiles if Chem.MolFromSmiles(s)]
    test_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius, nbits) for s in test_smiles if Chem.MolFromSmiles(s)]
    if not train_fps or not test_fps:
        return 0.0, 0.0
    sims = []
    for tf in test_fps:
        max_sim = max(DataStructs.TanimotoSimilarity(tf, tf_train) for tf_train in train_fps)
        sims.append(max_sim)
    return np.mean(sims), np.std(sims)


In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING WITH OPTUNA
# =============================================================================

def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 5e-4, log=True)
    hidden = trial.suggest_categorical('hidden', [128, 192, 256])
    heads = trial.suggest_categorical('heads', [4, 6, 8])
    dropout = trial.suggest_float('dropout', 0.2, 0.5)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    edge_dropout_p = trial.suggest_float('edge_dropout_p', 0.05, 0.3)
    feat_mask_p = trial.suggest_float('feat_mask_p', 0.05, 0.3)
    alpha = trial.suggest_float('alpha', 0.5, 0.9)
    gamma = trial.suggest_int('gamma', 1, 3)
    reg_weight = trial.suggest_float('reg_weight', 0.05, 0.2)

    # Split data
    idx = list(range(len(graphs)))
    labels_all = df_filtered['active'].values
    train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=labels_all)

    # Build loaders (no augmentation during tuning to save time)
    train_loader = DataLoader([graphs[i] for i in train_idx], batch_size = 128, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader([graphs[i] for i in val_idx], batch_size = 128, shuffle=False, num_workers=4, pin_memory=True)

    node_dim = graphs[0].x.shape[1]
    edge_dim = graphs[0].edge_attr.shape[1]

    model = UnifiedMSMP(node_dim, edge_dim, hidden=hidden, heads=heads, dropout=dropout).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20)

    cls_loss = FocalLoss(alpha=alpha, gamma=gamma)
    reg_loss = nn.MSELoss()

    best_val_acc = 0
    for epoch in range(12):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            c, r = model(batch)
            loss = cls_loss(c, batch.y_cls.squeeze()) + reg_weight * reg_loss(r, batch.y_reg_norm.squeeze())
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        probs, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                c, _ = model(batch)
                probs.extend(torch.sigmoid(c).cpu().numpy())
                labels.extend(batch.y_cls.cpu().numpy())
        fpr, tpr, thr = roc_curve(labels, probs)
        best_thr = thr[np.argmax(tpr - fpr)]
        preds = (np.array(probs) >= best_thr).astype(int)
        val_acc = accuracy_score(labels, preds)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
    return best_val_acc

print("\nStarting hyperparameter tuning (15 trials)...")
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=3, interval_steps=1))
study.optimize(objective, n_trials=25)
print("\nBest hyperparameters:")
print(study.best_trial.params)
best_hparams = study.best_trial.params

In [ ]:
# =============================================================================
# SPLIT GENERATION (RANDOM & SCAFFOLD)
# =============================================================================

def get_random_splits(n_splits=5, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    indices = np.arange(len(graphs))
    splits = []
    for train_idx, test_idx in kf.split(indices):
        splits.append((train_idx.tolist(), test_idx.tolist()))
    return splits

def get_scaffold_splits(df, n_splits=5):
    df = df.copy()
    df['scaffold'] = df['SMILES'].apply(
        lambda x: MurckoScaffold.MurckoScaffoldSmiles(smiles=x, includeChirality=True)
    )
    unique_scaffolds = df['scaffold'].unique()
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = []
    
    for train_scaff_idx, test_scaff_idx in kf.split(unique_scaffolds):
        train_scaffolds = unique_scaffolds[train_scaff_idx]
        test_scaffolds = unique_scaffolds[test_scaff_idx]
        
        train_idx = df[df['scaffold'].isin(train_scaffolds)].index.tolist()
        test_idx = df[df['scaffold'].isin(test_scaffolds)].index.tolist()
        
        splits.append((train_idx, test_idx))
    
    return splits

In [ ]:
# =============================================================================
# CROSS-VALIDATION FUNCTION WITH ENSEMBLE COLLECTION (and optional splits return)
# =============================================================================

def run_cv_ensemble(split_type, n_folds=5, hparams=None, n_ensemble=3):
    print(f"\n{'='*60}")
    print(f"RUNNING {split_type.upper()} SPLIT {n_folds}-FOLD CV WITH {n_ensemble}-MODEL ENSEMBLE")
    print('='*60)

    if split_type == 'random':
        splits = get_random_splits(n_splits=n_folds)
    else:
        splits = get_scaffold_splits(df_filtered, n_splits=n_folds)

    node_dim = graphs[0].x.shape[1]
    edge_dim = graphs[0].edge_attr.shape[1]

    all_cls_results = []
    all_reg_results = []
    all_similarities = []  # store train-test similarity per fold

    # Precompute regression normalization stats (global for simplicity)
    y_reg_all = np.array([g.y_reg.item() for g in graphs])
    y_reg_mean = y_reg_all.mean()
    y_reg_std = y_reg_all.std() + 1e-8
    # Normalize regression targets in graphs (will be used in loss)
    for g in graphs:
        g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std

    for fold, (train_idx, test_idx) in enumerate(splits):
        print(f"\n--- Fold {fold+1}/{n_folds} ---")

        # Train/Validation split (80/20 of training set)
        labels_train = [df_filtered['active'].iloc[i] for i in train_idx]
        train_subset, val_subset = train_test_split(train_idx, train_size=0.8, random_state=42+fold, stratify=labels_train)

        # ----- Data Augmentation: SMILES Randomization for training set -----
        train_smiles = [df_filtered['SMILES'].iloc[i] for i in train_subset]
        train_graphs_aug = []
        for idx in train_subset:
            smi = df_filtered['SMILES'].iloc[idx]
            orig = graphs[idx]
            train_graphs_aug.append(orig)
            rand_smiles = smiles_randomization(smi, n_random=3)
            for rsmi in rand_smiles:
                aug_g = smiles_to_graph(rsmi, orig.y_cls.item(), orig.y_reg.item())
                if aug_g is not None:
                    aug_g.y_reg_norm = (aug_g.y_reg - y_reg_mean) / y_reg_std
                    train_graphs_aug.append(aug_g)
        # Validation and test sets (no augmentation)
        val_graphs = [graphs[i] for i in val_subset]
        test_graphs = [graphs[i] for i in test_idx]

        # Compute train-test similarity for this fold
        test_smiles = [df_filtered['SMILES'].iloc[i] for i in test_idx]
        mean_sim, std_sim = fold_similarity(train_smiles, test_smiles)
        all_similarities.append((mean_sim, std_sim))
        print(f"  Train-test Tanimoto similarity: {mean_sim:.4f} ± {std_sim:.4f}")

        # Prepare data loaders
        train_loader = DataLoader(train_graphs_aug, batch_size = 128, shuffle=True, num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_graphs, batch_size = 128, shuffle=False, num_workers=4, pin_memory=True)
        test_loader  = DataLoader(test_graphs, batch_size = 128, shuffle=False, num_workers=4, pin_memory=True)

        # Train ensemble of n_ensemble models with different seeds
        fold_probs = []
        fold_reg_preds = []
        fold_labels = None
        fold_thresholds = []

        for seed in range(42, 42 + n_ensemble):
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = UnifiedMSMP(
                node_dim, edge_dim,
                hidden=hparams['hidden'],
                heads=hparams['heads'],
                dropout=hparams['dropout']
            ).to(device)

            best_model, best_thr = train_model_with_early_stopping(
                model, train_loader, val_loader, device,
                epochs=200, patience=20, augment=True,
                alpha=hparams['alpha'], gamma=hparams['gamma'],
                reg_weight=hparams['reg_weight'], lr=hparams['lr'],
                edge_dropout_p=hparams['edge_dropout_p'],
                feat_mask_p=hparams['feat_mask_p']
            )

            # Evaluate on test set
            model.eval()
            probs = []
            reg_preds = []
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(device)
                    c, r = best_model(batch)
                    probs.extend(torch.sigmoid(c).cpu().numpy())
                    reg_preds.extend(r.cpu().numpy())
            fold_probs.append(probs)
            fold_reg_preds.append(reg_preds)
            fold_thresholds.append(best_thr)

        # Average probabilities and regression outputs
        avg_probs = np.mean(fold_probs, axis=0)
        avg_reg = np.mean(fold_reg_preds, axis=0)
        # Use average threshold (or re-optimize on validation? We'll use mean of thresholds for simplicity)
        mean_thr = np.median(fold_thresholds)

        # Denormalize regression predictions
        avg_reg_denorm = avg_reg * y_reg_std + y_reg_mean
        true_reg = [graphs[i].y_reg.item() for i in test_idx]

        # Compute metrics
        preds = (avg_probs >= mean_thr).astype(int)
        labels = [graphs[i].y_cls.item() for i in test_idx]
        cm = confusion_matrix(labels, preds, labels=[0,1])
        tn, fp, fn, tp = cm.ravel()

        specificity = tn/(tn+fp) if (tn+fp)>0 else 0
        try:
            auc = roc_auc_score(labels, avg_probs)
        except:
            auc = 0.5
        cls_res = {
            'accuracy': accuracy_score(labels, preds),
            'auc': auc,
            'mcc': matthews_corrcoef(labels, preds),
            'sensitivity': recall_score(labels, preds),
            'specificity': specificity,
            'balanced_accuracy': balanced_accuracy_score(labels, preds),
            'f1': f1_score(labels, preds),
            'precision': precision_score(labels, preds),
            'recall': recall_score(labels, preds),
            'threshold': mean_thr
        }
        reg_res = {
            'r2': r2_score(true_reg, avg_reg_denorm),
            'rmse': np.sqrt(mean_squared_error(true_reg, avg_reg_denorm)),
            'mae': mean_absolute_error(true_reg, avg_reg_denorm)
        }

        print(f"Fold {fold+1} Test: ACC={cls_res['accuracy']:.4f}, AUC={cls_res['auc']:.4f}, MCC={cls_res['mcc']:.4f}")
        print(f"         R2={reg_res['r2']:.4f}, RMSE={reg_res['rmse']:.4f}, MAE={reg_res['mae']:.4f}")

        all_cls_results.append(cls_res)
        all_reg_results.append(reg_res)

    # Aggregate results
    agg_cls = {}
    for metric in all_cls_results[0].keys():
        values = [r[metric] for r in all_cls_results]
        agg_cls[metric] = {'mean': np.mean(values), 'std': np.std(values)}
    agg_reg = {}
    if all_reg_results:
        for metric in all_reg_results[0].keys():
            values = [r.get(metric, 0) for r in all_reg_results]
            agg_reg[metric] = {'mean': np.mean(values), 'std': np.std(values)}

    print("\n===== CROSS VALIDATION SUMMARY =====")
    for metric, vals in agg_cls.items():
        print(f"{metric}: {vals['mean']:.4f} ± {vals['std']:.4f}")
    for metric, vals in agg_reg.items():
        print(f"{metric}: {vals['mean']:.4f} ± {vals['std']:.4f}")
    print(f"\nTrain-test similarity across folds: mean {np.mean([s[0] for s in all_similarities]):.4f} ± {np.std([s[0] for s in all_similarities]):.4f}")

    return all_cls_results, all_reg_results, agg_cls, agg_reg, all_similarities


RUNNING SCAFFOLD SPLIT 5-FOLD CV

--- Fold 1/5 ---
Epoch 5: Val AUC = 0.9439, Val Acc (opt thr) = 0.8972, Thr = 0.35
Epoch 10: Val AUC = 0.9395, Val Acc (opt thr) = 0.8987, Thr = 0.47
Epoch 15: Val AUC = 0.9399, Val Acc (opt thr) = 0.8972, Thr = 0.42
Epoch 20: Val AUC = 0.9322, Val Acc (opt thr) = 0.8897, Thr = 0.40
Epoch 25: Val AUC = 0.9308, Val Acc (opt thr) = 0.8912, Thr = 0.49
Early stopping at epoch 26
Fold 1 Test: ACC=0.7827, AUC=0.8237, MCC=0.5642, Sens=0.7788, Spec=0.7874
         R2=0.1382, RMSE=1.1960, MAE=0.9434

--- Fold 2/5 ---
Epoch 5: Val AUC = 0.9152, Val Acc (opt thr) = 0.8557, Thr = 0.44
Epoch 10: Val AUC = 0.9109, Val Acc (opt thr) = 0.8659, Thr = 0.53
Epoch 15: Val AUC = 0.9032, Val Acc (opt thr) = 0.8703, Thr = 0.54
Epoch 20: Val AUC = 0.9046, Val Acc (opt thr) = 0.8703, Thr = 0.48
Epoch 25: Val AUC = 0.9014, Val Acc (opt thr) = 0.8703, Thr = 0.56
Early stopping at epoch 28
Fold 2 Test: ACC=0.8259, AUC=0.8783, MCC=0.6523, Sens=0.7882, Spec=0.8616
         R2=0.19

In [ ]:
# %%
# =============================================================================
# RUN RANDOM SPLIT CV WITH BEST HYPERPARAMETERS
# =============================================================================
random_cls, random_reg, random_agg_cls, random_agg_reg, random_sim = run_cv_ensemble('random', n_folds=5, hparams=best_hparams, n_ensemble=3)


In [ ]:
# %%
# =============================================================================
# RUN SCAFFOLD SPLIT CV WITH BEST HYPERPARAMETERS
# =============================================================================
scaffold_cls, scaffold_reg, scaffold_agg_cls, scaffold_agg_reg, scaffold_sim = run_cv_ensemble('scaffold', n_folds=5, hparams=best_hparams, n_ensemble=3)


STATISTICAL COMPARISON: RANDOM vs SCAFFOLD SPLIT
AUC: Random = 0.9169 ± 0.0054, Scaffold = 0.8790 ± 0.0319
Paired t-test: t=2.2622, p=8.6472e-02
MCC: Random = 0.7322 ± 0.0218, Scaffold = 0.6603 ± 0.0620
Paired t-test: t=2.3913, p=7.5062e-02

Results saved to gnn_cv_results_20260311_145735.json


: 

In [ ]:
# =============================================================================
# STATISTICAL COMPARISON (paired t-test)
# =============================================================================

print("\n" + "="*70)
print("STATISTICAL COMPARISON: RANDOM vs SCAFFOLD SPLIT")
print("="*70)

auc_random = [r['auc'] for r in random_cls]
auc_scaffold = [r['auc'] for r in scaffold_cls]
t_stat_auc, p_val_auc = stats.ttest_rel(auc_random, auc_scaffold)
print(f"AUC: Random = {np.mean(auc_random):.4f} ± {np.std(auc_random):.4f}")
print(f"AUC: Scaffold = {np.mean(auc_scaffold):.4f} ± {np.std(auc_scaffold):.4f}")
print(f"Paired t-test: t={t_stat_auc:.4f}, p={p_val_auc:.4e}")

mcc_random = [r['mcc'] for r in random_cls]
mcc_scaffold = [r['mcc'] for r in scaffold_cls]
t_stat_mcc, p_val_mcc = stats.ttest_rel(mcc_random, mcc_scaffold)
print(f"MCC: Random = {np.mean(mcc_random):.4f} ± {np.std(mcc_random):.4f}")
print(f"MCC: Scaffold = {np.mean(mcc_scaffold):.4f} ± {np.std(mcc_scaffold):.4f}")
print(f"Paired t-test: t={t_stat_mcc:.4f}, p={p_val_mcc:.4e}")